In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import math
import re
import torch
import numpy as np

In [2]:
FEATURE_COLUMNS = ['beta_propensity', 'proline_fraction', 'net_charge', 'polar_fraction', 'AAT', 'TA', 'a3vSA']

path = "files/df_ml_good_with_features.csv"
df = pd.read_csv(path)
df = df[df["class"] == 1].copy()

print(df.shape)
df.head()

(1006, 21)


,id,sequence,length,class,merge_id,a3vSA,AAT,Na4vSS,THSA,nHS,...,net_charge,hydrophobicity,polar_fraction,nonpolar_fraction,acidic_fraction,basic_fraction,aromatic_fraction,beta_propensity,seq_entropy,proline_fraction
0,GI_171848907_PDB_2RNM_A__1_1,GRNSAKDIRTEERARVQLGNV,21,1,GI_171848907_PDB_2RNM_A__1_1_0,-0.457,0.190,-0.494,0.190,1.0,...,2.0,-1.185714,0.238095,0.380952,0.142857,0.238095,0.000000,0.953333,3.535175,0.000000
1,GI_342871650_GB_EGU74155_1_1,VRIYAKDIKSEEMARVRVGNE,21,1,GI_342871650_GB_EGU74155_1_1_0,-0.160,2.140,-0.165,1.871,1.0,...,1.0,-0.676190,0.142857,0.428571,0.190476,0.238095,0.047619,0.990000,3.427333,0.000000
2,GI_342887385_GB_EGU86897_1_1,GKNSAGRINGPGMVNIGNS,19,1,GI_342887385_GB_EGU86897_1_1_0,-0.256,1.438,-0.256,1.438,1.0,...,2.0,-0.563158,0.315789,0.578947,0.000000,0.105263,0.000000,0.937368,3.005315,0.052632
3,GI_347837243_EMB_CCD5181_1_1,HRIKIGKVTQASNAKAVIGVH,21,1,GI_347837243_EMB_CCD5181_1_1_0,-0.001,4.054,-0.008,4.054,2.0,...,4.2,-0.019048,0.190476,0.523810,0.000000,0.285714,0.000000,1.081429,3.296148,0.000000
4,GI_475677570_GB_EMT74561_1_1,VRNYASEIKGDEDAKVRLGND,21,1,GI_475677570_GB_EMT74561_1_1_0,-0.436,0.570,-0.435,0.000,0.0,...,-1.0,-1.138095,0.190476,0.380952,0.238095,0.190476,0.047619,0.912381,3.499228,0.000000


In [3]:
MAX_LEN = 50

FEATURE_COLUMNS = [
    'beta_propensity',
    'proline_fraction',
    'net_charge',
    'polar_fraction'
]

# scaling (IMPORTANT: before dataset)
scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS])


PAD_IDX = 0

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i+1: aa for i, aa in enumerate(AMINO_ACIDS)}

EOS = len(AMINO_ACIDS) + 1
VOCAB_SIZE = EOS + 1


def encode_sequence(seq):
    seq = seq[:MAX_LEN-1]
    tokens = [aa_to_idx.get(a, PAD_IDX) for a in seq]
    tokens.append(EOS)
    tokens += [PAD_IDX] * (MAX_LEN - len(tokens))
    return torch.tensor(tokens)


def decode_sequence(tokens):
    out = []
    for t in tokens:
        if t == EOS:
            break
        if t != PAD_IDX:
            out.append(idx_to_aa[t])
    return "".join(out)


class ProteinDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        seq = encode_sequence(row["sequence"])

        features = torch.tensor(
            row[FEATURE_COLUMNS].astype(float).to_numpy(),
            dtype=torch.float32
        )

        return seq, features

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## Encoder, Decoder, cVAE

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, latent_dim, feature_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.len_proj = nn.Linear(1, 16)

        self.fc_mu = nn.Linear(d_model + 32 + 16, latent_dim)
        self.fc_logvar = nn.Linear(d_model + 32 + 16, latent_dim)

    def forward(self, x, features):
        x_emb = self.embedding(x)
        h = self.encoder(x_emb).mean(dim=1)

        f = self.feature_mlp(features)

        length = (x != PAD_IDX).sum(dim=1, keepdim=True).float()
        len_vec = self.len_proj(length)

        h = torch.cat([h, f, len_vec], dim=1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar, f, len_vec

In [6]:
def reparam(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [7]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, latent_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        self.latent_to_ctx = nn.Linear(latent_dim, d_model)
        self.len_to_ctx = nn.Linear(16, d_model)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=2)

        self.output = nn.Linear(d_model, vocab_size)

        self.memory_proj = nn.Linear(32, d_model)

    def forward(self, tgt, z, f_enc, len_vec, tgt_mask=None):

        tgt_emb = self.embedding(tgt)

        memory = self.memory_proj(f_enc)

        z_ctx = self.latent_to_ctx(z).unsqueeze(1)
        len_ctx = self.len_to_ctx(len_vec).unsqueeze(1)

        memory = memory + z_ctx + len_ctx

        out = self.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask
        )

        return self.output(out)

In [8]:
class ProteinCVAE(nn.Module):
    def __init__(self, vocab_size, feature_dim, d_model=128, latent_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)

        # ===== ENCODER =====
        enc_layer = nn.TransformerEncoderLayer(d_model, 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.feature_proj = nn.Linear(feature_dim, d_model)

        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_logvar = nn.Linear(d_model, latent_dim)

        # ===== DECODER =====
        dec_layer = nn.TransformerDecoderLayer(d_model, 4, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=2)

        self.latent_to_memory = nn.Linear(latent_dim, d_model)

        self.output = nn.Linear(d_model, vocab_size)

        # Ta warstwa automatycznie przyjmie wyższy feature_dim (np. 7)
        # i nauczy się go odtwarzać z wektora z
        self.feature_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim)
        )

    def encode(self, x, features):
        x = self.embedding(x)
        h = self.encoder(x).mean(dim=1)

        # Łączenie reprezentacji sekwencji z warunkiem (cechami)
        h = h + self.feature_proj(features)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def decode(self, tgt, z):
        tgt = self.embedding(tgt)

        memory = self.latent_to_memory(z).unsqueeze(1)  # [B,1,D]

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1)
        ).to(tgt.device)

        out = self.decoder(tgt, memory, tgt_mask=tgt_mask)

        return self.output(out)

    def forward(self, x, features):
        mu, logvar = self.encode(x, features)

        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)

        inp = x[:, :-1]
        tgt = x[:, 1:]

        logits = self.decode(inp, z)

        # Głowa predykcyjna zwraca wektor o długości równej liczbie cech
        pred_feat = self.feature_predictor(z)

        return logits, tgt, mu, logvar, pred_feat

In [9]:
def loss_fn(logits, target, mu, logvar, feat_pred, feat_true,
            beta=0.1, alpha=1.0):

    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1),
        ignore_index=PAD_IDX
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    feat_loss = F.mse_loss(feat_pred, feat_true)

    return recon + beta*kl + alpha*feat_loss, recon, kl, feat_loss

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ProteinCVAE(
    vocab_size=VOCAB_SIZE,
    feature_dim=len(FEATURE_COLUMNS)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [11]:
print(device)

cuda


In [12]:
# split danych
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# datasety
train_dataset = ProteinDataset(train_df)
val_dataset = ProteinDataset(val_df)

# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    drop_last=False
)

In [13]:
best_val = float("inf")
patience = 5
counter = 0
best_state = None
EPOCHS = 60

best_val_loss = float("inf")
counter = 0


def loss_fn(logits, target, mu, logvar, feat_pred, feat_true,
            beta=0.1, alpha=1.0):

    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1),
        ignore_index=PAD_IDX
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    feat_loss = F.mse_loss(feat_pred, feat_true)

    total = recon + beta * kl + alpha * feat_loss

    return total, recon, kl, feat_loss


for epoch in range(EPOCHS):

    beta = min(0.5, epoch / 20)

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for x, f in train_loader:
        x, f = x.to(device), f.to(device)

        optimizer.zero_grad()

        logits, tgt, mu, logvar, feat_pred = model(x, f)

        loss, recon, kl, feat = loss_fn(
            logits, tgt, mu, logvar,
            feat_pred, f,
            beta=beta,
            alpha=1.0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, f in val_loader:
            x, f = x.to(device), f.to(device)

            logits, tgt, mu, logvar, feat_pred = model(x, f)

            loss, _, _, _ = loss_fn(
                logits, tgt, mu, logvar,
                feat_pred, f,
                beta=beta,
                alpha=1.0
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1:03d} | train {train_loss:.4f} | val {val_loss:.4f} | beta {beta:.2f} | recon {recon:.2f} | kl {kl:.2f}")

    # ================= EARLY STOP =================
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_cvae.pt")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break


# ===== LOAD BEST MODEL =====
model.load_state_dict(torch.load("best_cvae.pt"))
torch.save(model.state_dict(), "./files/models/cvae_final.pt")

Epoch 001 | train 3.4940 | val 3.0584 | beta 0.00 | recon 2.86 | kl 0.83
Epoch 002 | train 2.9532 | val 2.8428 | beta 0.05 | recon 2.72 | kl 0.97
Epoch 003 | train 2.8222 | val 2.7814 | beta 0.10 | recon 2.68 | kl 0.78
Epoch 004 | train 2.7608 | val 2.7518 | beta 0.15 | recon 2.54 | kl 0.60
Epoch 005 | train 2.7173 | val 2.7181 | beta 0.20 | recon 2.54 | kl 0.49
Epoch 006 | train 2.6929 | val 2.7324 | beta 0.25 | recon 2.49 | kl 0.49
Epoch 007 | train 2.6600 | val 2.7196 | beta 0.30 | recon 2.54 | kl 0.45
Epoch 008 | train 2.6352 | val 2.7440 | beta 0.35 | recon 2.36 | kl 0.46
Epoch 009 | train 2.6055 | val 2.7158 | beta 0.40 | recon 2.47 | kl 0.42
Epoch 010 | train 2.5880 | val 2.7129 | beta 0.45 | recon 2.33 | kl 0.42
Epoch 011 | train 2.5761 | val 2.7262 | beta 0.50 | recon 2.32 | kl 0.39
Epoch 012 | train 2.5220 | val 2.7320 | beta 0.50 | recon 2.26 | kl 0.39
Epoch 013 | train 2.4659 | val 2.6853 | beta 0.50 | recon 2.25 | kl 0.38
Epoch 014 | train 2.4374 | val 2.6754 | beta 0.50 |

In [14]:
def generate(model, features, max_len=50, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        #features: [Feature_Dim] -> [1, Feature_Dim]
        features = features.unsqueeze(0).to(device)

        # Mapujemy cechy bezpośrednio na przestrzeń latentną tak, jak robi to model
        h_feat = model.feature_proj(features)
        mu = model.fc_mu(h_feat)
        z = mu # W czystej inferencji warunkowej przyjmujemy z = mu

        seq = torch.zeros((1, 1), dtype=torch.long).to(device) # Start od tokenu PAD/SOS

        for _ in range(max_len):
            logits = model.decode(seq, z)
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1)

            seq = torch.cat([seq, next_token], dim=1)

            if next_token.item() == EOS:
                break

        return decode_sequence(seq[0].cpu().numpy())

In [15]:
best_state = "./files/models/cvae_final.pt"
state_dict = torch.load(best_state)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [ ]:
df = pd.read_csv("./files/df_ml_good_with_features.csv")
df = df[df["class"] == 1].copy()

generated = []

N_ROWS = 1006
N_PER_ROW = 20
MAX_LEN = 50
device = next(model.parameters()).device


def perturb_features(row, noise_level=0.1):
    feats = row[FEATURE_COLUMNS].values.astype(float)

    noise = np.random.normal(0, noise_level, size=len(feats))
    feats = feats + noise

    feats = scaler.transform(feats.reshape(1, -1))[0]
    return torch.tensor(feats, dtype=torch.float32)


def generate_cvae(model, features, max_len=50, temperature=0.8):
    model.eval()

    with torch.no_grad():
        features = features.to(device)

        # Bezpośrednie przejście przez warstwy warunkowania cech (czyste mapowanie cech na 'z')
        h_feat = model.feature_proj(features.unsqueeze(0))
        mu = model.fc_mu(h_feat)

        # Twój mechanizm próbkowania z lekkim rozrzutem dla zachowania różnorodności
        z = mu + torch.randn_like(mu) * 0.1

        seq = torch.zeros((1, 1), dtype=torch.long).to(device)

        for _ in range(max_len):
            logits = model.decode(seq, z)
            probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
            token = torch.multinomial(probs, 1)

            seq = torch.cat([seq, token], dim=1)

            if token.item() == EOS:
                break

        return decode_sequence(seq[0].cpu().numpy())


# ================= MAIN LOOP =================

sample_df = df.sample(N_ROWS, random_state=42)

for _, row in sample_df.iterrows():

    for _ in range(N_PER_ROW):

        feats = perturb_features(row, noise_level=0.1)

        seq = generate_cvae(
            model,
            feats,
            max_len=MAX_LEN,
            temperature=0.8
        )

        # ===== CLEANING =====
        seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)

        if len(seq) >= 5:
            generated.append(seq)

In [20]:
df_out = pd.DataFrame({
    "sequence": generated,
    "length": [len(s) for s in generated]
})

df_out.to_csv(f"./files/generated_datases/CVAE_generated_sequences-{len(generated)}.csv", index=False)

print("Generated:", len(generated))

Generated: 18620


In [19]:
for i in range(100):
    print(f"{i} {generated[i]}")

0 PITVNF
1 VNSLQAGV
2 STVKAVQYGDM
3 PINLYIALN
4 VQSTGGKGTA
5 VKNIAEGVVITV
6 LNSPVIVLNVSSPI
7 VKDVQEGKARVGGNT
8 VSSRKSHVRG
9 LRPQAILANGNVTF
10 NIQGNGIN
11 LCVNPTVYVLSIIS
12 TSSTNRVVIHAGNSAE
13 ELRARSGPVGGNTNF
14 SLVAIT
15 DVNNSSLVTLQGSAKI
16 YISGEALSGQT
17 VTARLASGDGQSYIQ
18 LYNPPLNIIS
19 GQAKGLRVGGN
20 TVIPPAMPGDKLMSKIGVNI
21 PPIKLSGGGGVEI
22 PTAIPPKSGESVNV
23 HSSFLNAKNELYQTVGN
24 AGRSGRYQGRVE
25 NLYQANHCPSLIGN
26 RAGGVRYGNIKVRD
27 PGVNLNVFLQAKIGALM
28 SLVMGKAGNGLRVGDG
29 SRAGRVEHAKGHVQAK
30 HIPTSFVKSP
31 KNVSIGEE
32 VPPPPPISGIVLIISLNALNV
33 GRANTKFGDSEM
34 ILMTVPSKAVYGNKI
35 APQPNANLLVLVPSVFDGS
36 PSGSLDRSIVTLV
37 GTVQNSKGKAVSTQ
38 PILNVRLNEVNASGNVNAVNFLFQAAP
39 IKSAGT
40 SNKGKVMSGHGRVHNAKGG
41 VQSGDVAIKVGK
42 TAMHIGSKLAMQV
43 KGHQAEVKTASNGDKV
44 SKVLVV
45 ANPQGSIYLSSGGNFVNIT
46 TAMVKESIGRQ
47 PVNLSISQANFSNSKNAPQFLNAPQAYSPN
48 VHNSSAQVGN
49 LSAPSGAIHDQVQVYIHGGGG
50 VRSLRAGGDNT
51 RYEKGDVARSHLGD
52 NKELGRAAF
53 VKAEVTVIKVNLQGNIK
54 NPPPGGVLANVLVNMPVFL
55 TVKVTN
56 PGIANISS
57 PILIINAVQ